In [ ]:
# cs781 phase 1, model 2 on the cleaned corpus (gelu version): decoder only transformer (rope + gqa + gelu ffn, tied output head)
#
# same as the swiglu version, the only architecture change is the ffn: swiglu -> plain gelu mlp
#
# same setup as the model 1 notebook (window cache, checkpoints, resume, submission), but this one trains on the
# cleaned corpus made by bpe-final.ipynb and uses the byte level bpe from that notebook
#
# meant for a kaggle "save & commit" notebook (paste into one cell or run as a script)
# everything needed to survive a dead session goes in WORK_DIR/checkpoints:
#   latest.pt / latest_meta.json   full state (model, optimizer, scaler, position), always overwritten
#   best.pt / best_meta.json       weights of the best val loss so far
#   train_log.csv, progress.json   readable progress
# to resume after a session ends: add the previous version's output as an input of the next run
# and run again, the newest latest.pt of this model is picked up automatically
# (checkpoints have a model tag, so model 1 files that happen to be attached are ignored)
import os
import re
import csv
import sys
import json
import math
import time
import zlib
import shutil
import zipfile
import contextlib
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, list_repo_files, login

# config
MODEL_TAG = "decoder_only_rope_gqa_gelu_clean_bpe32k_v1"

IS_KAGGLE = os.path.isdir("/kaggle/working")
WORK_DIR = "/kaggle/working/cs781_phase1_model2_clean_gelu" if IS_KAGGLE else "./cs781_phase1_model2_clean_gelu"
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
CACHE_DIR = os.path.join(WORK_DIR, "cache_clean_bpe32k_v1")
INPUT_ROOT = "/kaggle/input"

# output files, same names and layout as the starter code
OUTPUT_DIR = "/kaggle/working/cs781_phase1" if IS_KAGGLE else "./cs781_phase1"
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

# data
# the bpe notebook has already run, this model uses the cleaned, deduplicated, source balanced corpus it made
# the code looks for clean_bpe_training.txt in the attached kaggle inputs, so no need to hard code a dataset name
# the same attached bpe dataset should also have tokenizer.json and meta.json
CLEAN_CORPUS_FILENAME = "clean_bpe_training.txt"

# hf is only used for the held out test set at prediction time
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
TEST_FILE_ALT = "Phase_1_Pretraining/Test/Phase1- Test.json"

# no raw SOURCES/MIX here anymore, the bpe notebook already built the clean corpus with
# supreme court 10%, high courts 50%, tribunals 35%, acts 5%

# new tag so a checkpoint from the old raw corpus run can't be resumed by mistake
MODEL_TAG = "decoder_only_rope_gqa_gelu_clean_bpe32k_v1"

# only paste a token here if you don't use the kaggle secret HF
# (the secret is better)
HF_TOKEN_FALLBACK = ""

# tokenizer
VOCAB_SIZE = 32_000

# sequence / dataset
CONTEXT_LENGTH = 1024
STRIDE = 768
WINDOW = CONTEXT_LENGTH + 1        # 1024 inputs + 1 extra so the labels can be shifted by one
MAX_WINDOWS = 400_000              # training windows per epoch
VAL_WINDOWS = 2_000
MAX_WIN_PER_DOC = 8
VAL_MAX_WIN_PER_DOC = 2
VAL_MOD = 40                       # doc is val if crc32(source:idx) % VAL_MOD == 0
MIN_DOC_CHARS = 4000

# transformer

D_MODEL = 640
N_LAYERS = 8
N_HEADS = 10
N_KV_HEADS = 2
HEAD_DIM = 64
FFN_HIDDEN = 2048
DROPOUT = 0.2

# optimisation
EPOCHS = 5
MICRO_BS = 8                       # use 4 (and ACCUM 8) if you get cuda oom
ACCUM = 4
EFF_BS = MICRO_BS * ACCUM
LR = 3e-4
PEAK_LR = LR
MIN_LR_RATIO = 0.1
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.10
GRAD_CLIP = 1.0
BETAS = (0.9, 0.95)
SEED = 1234

LOG_EVERY = 50          # optimizer steps
EVAL_EVERY = 1000       # optimizer steps (val loss, maybe a new best)
SAVE_EVERY = 500        # optimizer steps (mid epoch "latest" checkpoint)
MAX_RUNTIME_H = 11.3    # stop (after saving) before kaggle's 12h limit

# generation
MAX_NEW_TOKENS = 256
PROMPT_TOKENS = CONTEXT_LENGTH - MAX_NEW_TOKENS     # prompt keeps its last 768 tokens
TARGET_WORDS = 128
TEMPERATURE = 0.5
TOP_K = 40
TOP_P = 0.90
REPETITION_PENALTY = 1.10
NO_REPEAT_NGRAM_SIZE = 3
RUN_PREDICTION = True
PREDICT_ONLY = False       # True = skip training and predict from the best checkpoint
RUN_SELF_TEST = True       # cached decoding vs plain forward pass, runs once at start (takes seconds)


AMP = {"enabled": False, "dtype": torch.float16}


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])


def log(*a):
    print(*a, flush=True)


def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"


# finding files
def walk_find(filename, roots):
    # all dirs under roots that contain `filename`
    hits = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if filename in files:
                hits.append(d)
    return hits


def hf_token():
    # use the kaggle secret named HF, no token is hard coded here
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF")
        if tok:
            return tok
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or HF_TOKEN_FALLBACK or None


def hf_login():
    # hf login, token from env HF_TOKEN or the kaggle secret HF
    tok = hf_token()
    if tok:
        os.environ["HF_TOKEN"] = tok
        login(token=tok)
        log("Hugging Face login successful.")
    else:
        log("warning: no HF token found (set the Kaggle secret 'HF'); trying anonymous access")
    return tok


# tokenizer (byte level bpe loader for model 2)

from tokenizers import Tokenizer


class LegalByteBPE:

    def __init__(
        self,
        tokenizer_json,
        meta_path=None
    ):

        self.tokenizer = Tokenizer.from_file(
            tokenizer_json
        )

        self.vocab_size = (
            self.tokenizer.get_vocab_size()
        )

        vocab = self.tokenizer.get_vocab()

        self.pad_id = vocab["<|pad|>"]
        self.unk_id = vocab["<|unk|>"]
        self.eos_id = vocab["<|endoftext|>"]

        self.bos_id = self.eos_id

        self.special_ids = {
            self.pad_id,
            self.unk_id,
            self.eos_id,
        }

        if meta_path is not None:

            with open(
                meta_path,
                "r",
                encoding="utf-8"
            ) as f:

                meta = json.load(f)

            assert meta["vocab_size"] == self.vocab_size

            assert meta["pad_id"] == self.pad_id
            assert meta["unk_id"] == self.unk_id
            assert meta["eos_id"] == self.eos_id

    def encode(self, text):

        return self.tokenizer.encode(
            text
        ).ids

    def decode(self, ids):

        return self.tokenizer.decode(
            ids,
            skip_special_tokens=True
        )

def clean_text(s):
    # glue punctuation back onto words (the bpe gives every punctuation group as its own "word")
    s = re.sub(r"\s+([.,;:!?)\]])", r"\1", s)
    s = re.sub(r"([(\[])\s+", r"\1", s)
    return re.sub(r"\s+", " ", s).strip()


def to_words(ids, tok, n_words=TARGET_WORDS):
    words = clean_text(tok.decode(ids)).split()
    return " ".join(words[:n_words])


def load_tokenizer():
    # take tokenizer.json from the same attached dataset as the clean corpus when possible,
    # so we don't pick up an older tokenizer from some other kaggle input
    preferred_root = os.path.dirname(CLEAN_CORPUS_PATH)

    roots = [
        preferred_root,
        INPUT_ROOT,
        os.getcwd(),
        WORK_DIR,
    ]

    seen = set()

    for root in roots:

        if not os.path.isdir(root):
            continue

        for d, _, files in os.walk(root):

            if d in seen:
                continue

            seen.add(d)

            if "tokenizer.json" not in files:
                continue

            tokenizer_json = os.path.join(
                d,
                "tokenizer.json"
            )

            meta_path = os.path.join(
                d,
                "meta.json"
            )

            log(
                f"byte-level tokenizer found in {d}"
            )

            tok = LegalByteBPE(
                tokenizer_json,
                meta_path if os.path.isfile(meta_path) else None,
            )

            log(
                f"vocab={tok.vocab_size} "
                f"pad={tok.pad_id} "
                f"unk={tok.unk_id} "
                f"eos={tok.eos_id}"
            )

            return tok

    raise FileNotFoundError(
        "tokenizer.json not found in the attached BPE dataset "
        "or other searched locations."
    )


# streaming reader for the already cleaned bpe corpus
def find_clean_corpus():
    # find the clean corpus made by the bpe notebook
    # the attached dataset should have clean_bpe_training.txt, tokenizer.json and meta.json
    # the corpus is never loaded fully into ram
    roots = [INPUT_ROOT, os.getcwd(), WORK_DIR]

    hits = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            if CLEAN_CORPUS_FILENAME in files:
                hits.append(os.path.join(d, CLEAN_CORPUS_FILENAME))

    # use the attached kaggle input before a copy in the working dir
    hits.sort(
        key=lambda p: (
            0 if p.startswith(INPUT_ROOT) else 1,
            len(p),
        )
    )

    if not hits:
        raise FileNotFoundError(
            f"{CLEAN_CORPUS_FILENAME!r} was not found.\n"
            "Attach the dataset produced by bpe-final.ipynb to this "
            "Model-2 notebook."
        )

    path = hits[0]
    size_gb = os.path.getsize(path) / (1024 ** 3)

    log(f"clean LM corpus found: {path}")
    log(f"clean LM corpus size : {size_gb:.2f} GB")

    return path


CLEAN_CORPUS_PATH = find_clean_corpus()


def stream_clean_documents(path):
    # stream the docs written by the bpe notebook
    # it writes every cleaned doc followed by a blank line, so a blank line is the separator
    # only one doc is held in memory at a time
    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as f:

        buf = []

        for line in f:

            if line.strip() == "":
                if buf:
                    text = "".join(buf).strip()
                    if text:
                        yield text
                    buf = []
            else:
                buf.append(line)

        if buf:
            text = "".join(buf).strip()
            if text:
                yield text


# build the train / val windows from the clean corpus
def build_windows(tok):
    # build lm windows straight from clean_bpe_training.txt
    # the source mix was already enforced by the bpe notebook (sc 10%, hc 50%, tribunals 35%, acts 5%), so no MIX here
    train_out, val_out = [], []

    t0 = time.time()
    n_docs = 0
    n_skipped = 0
    n_train = 0
    n_val = 0

    log("=" * 80)
    log("BUILDING WINDOWS FROM CLEAN BPE CORPUS")
    log("=" * 80)
    log(f"corpus       : {CLEAN_CORPUS_PATH}")
    log(f"target train : {MAX_WINDOWS:,}")
    log(f"target val   : {VAL_WINDOWS:,}")

    for text in stream_clean_documents(CLEAN_CORPUS_PATH):

        doc_idx = n_docs
        n_docs += 1

        if len(text) < MIN_DOC_CHARS:
            n_skipped += 1
            continue

        if n_train >= MAX_WINDOWS and n_val >= VAL_WINDOWS:
            break

        # deterministic val split over the cleaned docs
        # no source label here, the corpus was already balanced before bpe training
        is_val = (
            zlib.crc32(
                f"clean:{doc_idx}".encode()
            ) % VAL_MOD == 0
        )

        if is_val:
            if n_val >= VAL_WINDOWS:
                continue
            cap = VAL_MAX_WIN_PER_DOC
        else:
            if n_train >= MAX_WINDOWS:
                continue
            cap = MAX_WIN_PER_DOC

        ids = tok.encode(text)

        if len(ids) < WINDOW:
            continue

        wins = cut_windows(ids, cap)

        for w in wins:
            arr = np.asarray(w, dtype=np.uint16)

            if is_val:
                if n_val < VAL_WINDOWS:
                    val_out.append(arr)
                    n_val += 1
            else:
                if n_train < MAX_WINDOWS:
                    train_out.append(arr)
                    n_train += 1

        if n_docs % 2000 == 0:
            elapsed = time.time() - t0
            log(
                f"docs={n_docs:,} | "
                f"train={n_train:,}/{MAX_WINDOWS:,} | "
                f"val={n_val:,}/{VAL_WINDOWS:,} | "
                f"skipped={n_skipped:,} | "
                f"time={elapsed/60:.1f} min"
            )

    elapsed = time.time() - t0

    log("=" * 80)
    log("WINDOW BUILD COMPLETE")
    log("=" * 80)
    log(f"documents processed : {n_docs:,}")
    log(f"documents skipped   : {n_skipped:,}")
    log(f"train windows       : {n_train:,}/{MAX_WINDOWS:,}")
    log(f"validation windows  : {n_val:,}/{VAL_WINDOWS:,}")
    log(f"time                : {elapsed/60:.1f} min")

    if not train_out:
        raise RuntimeError(
            "No training windows were built. "
            "Check the attached clean_bpe_training.txt."
        )

    if not val_out:
        raise RuntimeError(
            "No validation windows were built. "
            "Check the clean_bpe_training.txt."
        )

    if n_train < MAX_WINDOWS:
        log(
            f"WARNING: only {n_train:,} train windows were available; "
            f"requested {MAX_WINDOWS:,}."
        )

    if n_val < VAL_WINDOWS:
        log(
            f"WARNING: only {n_val:,} validation windows were available; "
            f"requested {VAL_WINDOWS:,}."
        )

    return np.stack(train_out), np.stack(val_out)


def get_windows(tok):
    # cache signature has the clean corpus name and a new cache kind,
    # so windows from the old raw corpus never get reused
    sig = {
        "kind": "clean_bpe_training_v1",
        "corpus": os.path.basename(CLEAN_CORPUS_PATH),
        "context": CONTEXT_LENGTH,
        "stride": STRIDE,
        "train_windows": MAX_WINDOWS,
        "val_windows": VAL_WINDOWS,
        "vocab": tok.vocab_size,
        "min_chars": MIN_DOC_CHARS,
        "max_win": MAX_WIN_PER_DOC,
        "val_max_win": VAL_MAX_WIN_PER_DOC,
        "val_mod": VAL_MOD,
    }

    for d in walk_find(
        "windows_manifest.json",
        [CACHE_DIR, INPUT_ROOT],
    ):
        try:
            with open(
                os.path.join(d, "windows_manifest.json")
            ) as f:
                man = json.load(f)

            if man.get("sig") != sig:
                continue

            tr = np.load(
                os.path.join(d, "train_windows.npy")
            )
            va = np.load(
                os.path.join(d, "val_windows.npy")
            )

        except Exception as e:
            log(f"cache in {d} unusable ({e})")
            continue

        log(
            f"loaded cached clean-corpus windows from {d}: "
            f"train={tr.shape} val={va.shape}"
        )

        if os.path.abspath(d) != os.path.abspath(CACHE_DIR):
            save_windows(tr, va, sig)

        return tr, va

    log(
        "no clean-corpus window cache found; "
        "tokenizing clean_bpe_training.txt ..."
    )

    tr, va = build_windows(tok)
    save_windows(tr, va, sig)

    log(
        f"built windows: train={tr.shape} val={va.shape}"
    )

    return tr, va


def save_windows(tr, va, sig):
    os.makedirs(CACHE_DIR, exist_ok=True)

    np.save(
        os.path.join(CACHE_DIR, "train_windows.npy"),
        tr
    )

    np.save(
        os.path.join(CACHE_DIR, "val_windows.npy"),
        va
    )

    with open(
        os.path.join(CACHE_DIR, "windows_manifest.json"),
        "w",
    ) as f:
        json.dump(
            {
                "sig": sig,
                "train": len(tr),
                "val": len(va),
            },
            f,
        )


# sliding windows
def cut_windows(ids, max_n):
    # overlapping WINDOW token windows, one every STRIDE tokens from the start of the doc
    out, start, n = [], 0, len(ids)
    while len(out) < max_n and start + WINDOW <= n:
        out.append(ids[start:start + WINDOW])
        start += STRIDE
    return out


# model
@dataclass
class ModelConfig:
    vocab_size: int = VOCAB_SIZE
    d_model: int = D_MODEL
    n_layers: int = N_LAYERS
    n_heads: int = N_HEADS
    n_kv_heads: int = N_KV_HEADS
    ffn_hidden: int = FFN_HIDDEN
    dropout: float = DROPOUT
    max_len: int = CONTEXT_LENGTH + 64
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)


def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()          # each (max_len, head_dim/2)


def apply_rope(x, cos, sin):
    # x: (B, H, T, hd), cos/sin: (1 or B, 1, T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


def repeat_kv(x, groups):
    # (B, KV, T, hd) -> (B, KV*groups, T, hd), query head h uses kv head h // groups
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)


class GQAttention(nn.Module):
    # causal grouped query attention with rope (10 query heads share 2 kv heads)

    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def _qkv(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        return q, k, v

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

    def forward_cached(self, x, cos, sin, k_cache, v_cache, start, attn_mask):
        # the T new tokens go into cache slots [start, start+T) and attend to slots [0, start+T)
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        k_cache[:, :, start:start + T] = k.to(k_cache.dtype)
        v_cache[:, :, start:start + T] = v.to(v_cache.dtype)
        keys = repeat_kv(k_cache[:, :, :start + T], self.groups).to(q.dtype)
        vals = repeat_kv(v_cache[:, :, :start + T], self.groups).to(q.dtype)
        out = F.scaled_dot_product_attention(q, keys, vals, attn_mask=attn_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


class FFN(nn.Module):
    # plain gelu mlp (fc1 -> gelu -> fc2), no gate, no bias
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.fc2 = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = FFN(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

    def forward_cached(self, x, cos, sin, cache, start, attn_mask):
        x = x + self.attn.forward_cached(self.norm1(x), cos, sin, cache["k"], cache["v"], start, attn_mask)
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)       # also the tied output projection
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # scale down the residual branch output projections, like gpt2
        scale = 0.02 / math.sqrt(2 * cfg.n_layers)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("fc2.weight"):
                nn.init.normal_(p, mean=0.0, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def _rope(self, T, pos_ids=None):
        if pos_ids is None:
            return self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        return self.rope_cos[pos_ids][:, None], self.rope_sin[pos_ids][:, None]   # (B, 1, T, hd/2)

    def forward(self, idx):
        # training / val path: idx (B, T) -> logits (B, T, V)
        cos, sin = self._rope(idx.size(1))
        x = self.drop(self.embed(idx))
        for blk in self.blocks:
            x = blk(x, cos, sin)
        x = self.final_norm(x)
        return F.linear(x, self.embed.weight)           # weight tying: logits = h @ E^T

    # kv cached generation
    def init_cache(self, B, total_len, dtype, device):
        kv, hd = self.cfg.n_kv_heads, self.cfg.d_model // self.cfg.n_heads
        return [{"k": torch.zeros(B, kv, total_len, hd, dtype=dtype, device=device),
                 "v": torch.zeros(B, kv, total_len, hd, dtype=dtype, device=device)}
                for _ in self.blocks]

    def forward_cached(self, idx, pos_ids, cache, start, attn_mask):
        # idx (B, T) new tokens with rope positions pos_ids (B, T), returns the logits of the last one
        cos, sin = self._rope(idx.size(1), pos_ids)
        x = self.embed(idx)
        for blk, c in zip(self.blocks, cache):
            x = blk.forward_cached(x, cos, sin, c, start, attn_mask)
        x = self.final_norm(x)
        return F.linear(x[:, -1], self.embed.weight)


def count_params(model):
    return sum(p.numel() for p in model.parameters())      # tied weights are counted once


# decoding (batched, kv cache, left padded prompts)
def make_prompt_batch(seqs, pad_id, device):
    # left pad token lists into one tensor, mask is True on real tokens
    width = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), width), pad_id, dtype=torch.long)
    mask = torch.zeros(len(seqs), width, dtype=torch.bool)
    for r, s in enumerate(seqs):
        ids[r, width - len(s):] = torch.tensor(s, dtype=torch.long)
        mask[r, width - len(s):] = True
    return ids.to(device), mask.to(device)


def start_generation(model, prompt, prompt_mask, max_new):
    # prefill the cache with the left padded prompt, returns cache, last position logits, lengths, key validity
    B, W = prompt.shape
    dev = prompt.device
    lengths = prompt_mask.sum(1)                                        # real tokens per row
    pos_ids = (torch.arange(W, device=dev)[None] - (W - lengths)[:, None]).clamp(min=0)
    causal = torch.tril(torch.ones(W, W, dtype=torch.bool, device=dev))
    eye = torch.eye(W, dtype=torch.bool, device=dev)
    # pad rows can still see themselves, otherwise their softmax row is empty (nan)
    mask = (causal[None, None] & prompt_mask[:, None, None, :]) | eye[None, None]
    valid = torch.cat([prompt_mask, torch.ones(B, max_new, dtype=torch.bool, device=dev)], dim=1)
    cdtype = AMP["dtype"] if AMP["enabled"] else torch.float32
    cache = model.init_cache(B, W + max_new, cdtype, dev)
    logits = model.forward_cached(prompt, pos_ids, cache, 0, mask)
    return cache, logits, lengths, valid


def next_logits(model, tok, t, W, cache, lengths, valid):
    # feed the token from step t (it sits in cache slot W+t) and get the next logits
    pos = (lengths + t)[:, None]
    mask = valid[:, None, None, :W + t + 1]
    return model.forward_cached(tok[:, None], pos, cache, W + t, mask)


@torch.no_grad()
def generate_ids(model, prompt, prompt_mask, max_new=MAX_NEW_TOKENS, temperature=0.0, top_k=0,
                 top_p=1.0, rep_penalty=1.0, no_repeat_ngram=0):
    # temperature <= 0 means greedy, returns (B, max_new) generated token ids
    model.eval()
    cfg = model.cfg
    B, W = prompt.shape
    out = torch.zeros(B, max_new, dtype=torch.long, device=prompt.device)
    gen_lists = [[] for _ in range(B)]
    tables = [dict() for _ in range(B)]
    n = no_repeat_ngram
    with amp_ctx():
        cache, logits, lengths, valid = start_generation(model, prompt, prompt_mask, max_new)
        for t in range(max_new):
            logits = logits.float()
            logits[:, cfg.eos_id] = float("-inf")       # always generate the full continuation
            logits[:, cfg.pad_id] = float("-inf")
            logits[:, cfg.unk_id] = float("-inf")
            if rep_penalty != 1.0 and t > 0:
                prev = out[:, :t]
                s = logits.gather(1, prev)
                s = torch.where(s > 0, s / rep_penalty, s * rep_penalty)
                logits.scatter_(1, prev, s)
            if n > 1 and t >= n - 1:
                for b in range(B):
                    banned = tables[b].get(tuple(gen_lists[b][-(n - 1):]))
                    if banned:
                        logits[b, list(banned)] = float("-inf")
            if temperature <= 0:
                tok = logits.argmax(-1)
            else:
                logits = logits / temperature
                if top_k > 0:
                    k = min(top_k, logits.size(-1))
                    kth = torch.topk(logits, k).values[:, -1, None]
                    logits = logits.masked_fill(logits < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(logits, descending=True, dim=-1)
                    probs = sl.softmax(-1)
                    drop = (probs.cumsum(-1) - probs) > top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    logits = torch.full_like(logits, float("-inf")).scatter(1, si, sl)
                tok = torch.multinomial(logits.softmax(-1), 1).squeeze(1)
            out[:, t] = tok
            if n > 1:
                for b, x in enumerate(tok.tolist()):
                    g = gen_lists[b]
                    g.append(x)
                    if len(g) >= n:
                        tables[b].setdefault(tuple(g[-n:-1]), set()).add(x)
            if t < max_new - 1:
                logits = next_logits(model, tok, t, W, cache, lengths, valid)
    return out


@torch.no_grad()
def self_test(model, val_w, device):
    # the cached + left padded decoding path should give the same result as a plain full forward pass (fp32)
    model.eval()
    saved = AMP["enabled"]
    AMP["enabled"] = False
    try:
        rows = val_w[:3].astype(np.int64).tolist()
        seqs = [r[:L] for r, L in zip(rows, (300, 520, 700))]
        ids, mask = make_prompt_batch(seqs, model.cfg.pad_id, device)
        W = ids.size(1)
        steps = 3
        cache, logits, lengths, valid = start_generation(model, ids, mask, steps)
        for t in range(steps):
            ref = torch.stack([model(torch.tensor([s], dtype=torch.long, device=device))[0, -1] for s in seqs])
            err = (logits.float() - ref.float()).abs().max().item()
            if err > 5e-3:
                raise RuntimeError(f"KV-cache self-test failed at step {t}: max logit diff {err:.5f}")
            nxt = ref.argmax(-1)
            seqs = [s + [int(x)] for s, x in zip(seqs, nxt.tolist())]
            if t < steps - 1:
                logits = next_logits(model, nxt, t, W, cache, lengths, valid)
        log(f"self-test passed: cached left-padded decoding matches the full forward pass (last diff {err:.2e})")
    finally:
        AMP["enabled"] = saved


def rouge_l_f1(ref, hyp):
    a, b = ref.split(), hyp.split()
    if not a or not b:
        return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    lcs = prev[-1]
    if lcs == 0:
        return 0.0
    p, r = lcs / len(b), lcs / len(a)
    return 2 * p * r / (p + r)


def split_window(row, is_end=None):
    # split a val window at the configured prompt length
    # the byte level bpe has no word boundary table (tok.is_end), so a fixed token split is the only tokenizer independent option
    return min(PROMPT_TOKENS, len(row) - 1)


def val_gen_score(model, tok, val_w, device, n=64, bs=32, **dec_kwargs):
    # mean rouge-l f1 on the first n val windows (cheap stand-in for bertscore, which we can't compute offline)
    scores, samples = [], []
    n = min(n, len(val_w))
    for i in range(0, n, bs):
        rows = val_w[i:min(i + bs, n)].astype(np.int64).tolist()
        splits = [split_window(r) for r in rows]
        ids, mask = make_prompt_batch([r[:s] for r, s in zip(rows, splits)], tok.pad_id, device)
        gen = generate_ids(model, ids, mask, **dec_kwargs).cpu().tolist()
        for r, s, g in zip(rows, splits, gen):
            ref, hyp = to_words(r[s:s + MAX_NEW_TOKENS], tok), to_words(g, tok)
            scores.append(rouge_l_f1(ref, hyp))
            samples.append((ref, hyp))
    return float(np.mean(scores)), samples


# training helpers
def make_batch(windows, idx, device):
    w = torch.from_numpy(windows[idx].astype(np.int64))          # (B, CONTEXT_LENGTH + 1)
    return w[:, :-1].to(device), w[:, 1:].to(device)


@torch.no_grad()
def evaluate(model, windows, device, bs=16):
    model.eval()
    cfg = model.cfg
    total, count = 0.0, 0
    for i in range(0, len(windows), bs):
        idx = np.arange(i, min(i + bs, len(windows)))
        x, y = make_batch(windows, idx, device)
        with amp_ctx():
            logits = model(x)
        loss = F.cross_entropy(logits.float().reshape(-1, cfg.vocab_size), y.reshape(-1),
                               ignore_index=cfg.pad_id, reduction="sum")
        total += loss.item()
        count += (y != cfg.pad_id).sum().item()
    return total / max(count, 1)


def lr_at(step, total_steps):
    if step < WARMUP_STEPS:
        return PEAK_LR * (step + 1) / WARMUP_STEPS
    prog = min(1.0, (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS))
    return PEAK_LR * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * prog)))


def make_optimizer(model):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if p.ndim >= 2 and "embed" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    groups = [{"params": decay, "weight_decay": WEIGHT_DECAY},
              {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=PEAK_LR, betas=BETAS, eps=1e-8,
                             fused=torch.cuda.is_available())


def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def find_previous():
    # newest complete latest.pt (by global_step) in our dir or any attached kaggle input
    best = None
    for d in walk_find("latest_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "latest_meta.json"))
        if meta is None or meta.get("tag") != MODEL_TAG or not os.path.exists(os.path.join(d, "latest.pt")):
            continue
        if best is None or meta["global_step"] > best[1]["global_step"]:
            best = (d, meta)
    return best


def carry_over(src_dir):
    # copy the best checkpoint + log from an older run's output into this run's checkpoint dir
    if os.path.abspath(src_dir) == os.path.abspath(CKPT_DIR):
        return
    os.makedirs(CKPT_DIR, exist_ok=True)
    old_best = read_json(os.path.join(src_dir, "best_meta.json"))
    cur_best = read_json(os.path.join(CKPT_DIR, "best_meta.json"))
    if old_best and os.path.exists(os.path.join(src_dir, "best.pt")):
        if cur_best is None or old_best["val_loss"] < cur_best["val_loss"]:
            shutil.copy(os.path.join(src_dir, "best.pt"), os.path.join(CKPT_DIR, "best.pt.tmp"))
            os.replace(os.path.join(CKPT_DIR, "best.pt.tmp"), os.path.join(CKPT_DIR, "best.pt"))
            shutil.copy(os.path.join(src_dir, "best_meta.json"), os.path.join(CKPT_DIR, "best_meta.json"))
    src_log, dst_log = os.path.join(src_dir, "train_log.csv"), os.path.join(CKPT_DIR, "train_log.csv")
    if os.path.exists(src_log) and not os.path.exists(dst_log):
        shutil.copy(src_log, dst_log)


LOG_FIELDS = ["time", "kind", "epoch", "step_in_epoch", "global_step", "train_loss", "val_loss",
              "val_ppl", "lr", "rouge_l", "elapsed_h"]


def append_log(row):
    path = os.path.join(CKPT_DIR, "train_log.csv")
    new = not os.path.exists(path)
    with open(path, "a", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new:
            wr.writeheader()
        wr.writerow({k: row.get(k, "") for k in LOG_FIELDS})


# training loop
def train(model, tok, train_w, val_w, device, t_start):
    cfg = model.cfg
    os.makedirs(CKPT_DIR, exist_ok=True)
    n_train = len(train_w)
    steps_per_epoch = n_train // EFF_BS
    total_steps = EPOCHS * steps_per_epoch
    opt = make_optimizer(model)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP["enabled"] and AMP["dtype"] == torch.float16)
    state = {"epoch": 0, "step_in_epoch": 0, "global_step": 0, "best_val": float("inf")}

    prev = find_previous()
    if prev is not None:
        d, meta = prev
        log(f"resuming from {d}: epoch {meta['epoch'] + 1}, step {meta['step_in_epoch']}, global step {meta['global_step']}")
        ckpt = torch.load(os.path.join(d, "latest.pt"), map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model"])
        opt.load_state_dict(ckpt["optimizer"])
        scaler.load_state_dict(ckpt["scaler"])
        state = {k: ckpt[k] for k in state}
        del ckpt
        carry_over(d)
    else:
        log("no previous checkpoint found, starting from scratch")
    best_meta = read_json(os.path.join(CKPT_DIR, "best_meta.json"))
    if best_meta:
        state["best_val"] = min(state["best_val"], best_meta["val_loss"])

    if state["epoch"] >= EPOCHS:
        log("all epochs already finished in an earlier session")
        return True

    def save_latest(epoch, step_in_epoch):
        payload = {"model": model.state_dict(), "optimizer": opt.state_dict(), "scaler": scaler.state_dict(),
                   "epoch": epoch, "step_in_epoch": step_in_epoch, "global_step": state["global_step"],
                   "best_val": state["best_val"], "config": asdict(cfg), "tag": MODEL_TAG}
        atomic_torch_save(payload, os.path.join(CKPT_DIR, "latest.pt"))
        atomic_json({"epoch": epoch, "step_in_epoch": step_in_epoch, "global_step": state["global_step"],
                     "best_val": state["best_val"], "tag": MODEL_TAG, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")},
                    os.path.join(CKPT_DIR, "latest_meta.json"))

    def run_validation(epoch, step_in_epoch, last_loss):
        vl = evaluate(model, val_w, device)
        improved = vl < state["best_val"]
        if improved:
            state["best_val"] = vl
            atomic_torch_save({"model": model.state_dict(), "val_loss": vl, "epoch": epoch,
                               "step_in_epoch": step_in_epoch, "global_step": state["global_step"],
                               "config": asdict(cfg), "tag": MODEL_TAG}, os.path.join(CKPT_DIR, "best.pt"))
            atomic_json({"tag": MODEL_TAG, "val_loss": vl, "val_ppl": math.exp(vl), "epoch": epoch,
                         "step_in_epoch": step_in_epoch, "global_step": state["global_step"]},
                        os.path.join(CKPT_DIR, "best_meta.json"))
        tag = "(NEW BEST, saved best.pt)" if improved else "(best so far %.4f)" % state["best_val"]
        log(f"  >> validation: loss {vl:.4f} ppl {math.exp(vl):.2f} {tag}")
        append_log({"time": time.strftime("%H:%M:%S"), "kind": "val", "epoch": epoch + 1,
                    "step_in_epoch": step_in_epoch, "global_step": state["global_step"],
                    "train_loss": f"{last_loss:.4f}", "val_loss": f"{vl:.4f}", "val_ppl": f"{math.exp(vl):.2f}",
                    "elapsed_h": f"{(time.time() - t_start) / 3600:.2f}"})
        model.train()
        return vl

    def out_of_time():
        return (time.time() - t_start) > MAX_RUNTIME_H * 3600

    session_first_step = state["global_step"]
    session_t0 = time.time()
    log(f"steps/epoch={steps_per_epoch} total steps={total_steps} effective batch={EFF_BS} windows "
        f"({EFF_BS * CONTEXT_LENGTH:,} tokens/step)")

    model.train()
    for epoch in range(state["epoch"], EPOCHS):
        perm = np.random.default_rng(SEED + epoch).permutation(n_train)   # same order after resuming
        first = state["step_in_epoch"] if epoch == state["epoch"] else 0
        run_loss, run_n, last_loss = 0.0, 0, float("nan")
        t_log = time.time()
        for step in range(first, steps_per_epoch):
            lr = lr_at(state["global_step"], total_steps)
            for g in opt.param_groups:
                g["lr"] = lr
            batch_ids = perm[step * EFF_BS:(step + 1) * EFF_BS]
            step_loss = 0.0
            for m in range(ACCUM):
                idx = batch_ids[m * MICRO_BS:(m + 1) * MICRO_BS]
                x, y = make_batch(train_w, idx, device)
                with amp_ctx():
                    logits = model(x)
                loss = F.cross_entropy(logits.float().reshape(-1, cfg.vocab_size), y.reshape(-1),
                                       ignore_index=cfg.pad_id)
                scaler.scale(loss / ACCUM).backward()
                step_loss += loss.item() / ACCUM
            scaler.unscale_(opt)
            gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

            state["global_step"] += 1
            done = step + 1
            if math.isfinite(step_loss):
                run_loss += step_loss
                run_n += 1
                last_loss = step_loss

            if state["global_step"] % LOG_EVERY == 0 or done == steps_per_epoch:
                now = time.time()
                sess_steps = state["global_step"] - session_first_step
                sec_per_step = (now - session_t0) / max(sess_steps, 1)
                remaining = total_steps - state["global_step"]
                avg = run_loss / max(run_n, 1)
                overall = 100.0 * state["global_step"] / total_steps
                log(f"[ep {epoch + 1}/{EPOCHS}] step {done}/{steps_per_epoch} ({100 * done / steps_per_epoch:.1f}%) | "
                    f"overall {overall:.1f}% | loss {avg:.4f} | gnorm {float(gnorm):.2f} | lr {lr:.2e} | "
                    f"{EFF_BS * LOG_EVERY / max(now - t_log, 1e-9):.1f} win/s | "
                    f"elapsed {fmt_time(now - t_start)} | ETA {fmt_time(remaining * sec_per_step)}")
                atomic_json({"epoch": epoch + 1, "epochs": EPOCHS, "step_in_epoch": done,
                             "steps_per_epoch": steps_per_epoch, "global_step": state["global_step"],
                             "total_steps": total_steps, "overall_percent": round(overall, 2),
                             "train_loss": avg, "best_val_loss": state["best_val"],
                             "eta_this_session_h": round(remaining * sec_per_step / 3600, 2),
                             "updated": time.strftime("%Y-%m-%d %H:%M:%S")},
                            os.path.join(CKPT_DIR, "progress.json"))
                append_log({"time": time.strftime("%H:%M:%S"), "kind": "train", "epoch": epoch + 1,
                            "step_in_epoch": done, "global_step": state["global_step"],
                            "train_loss": f"{avg:.4f}", "lr": f"{lr:.3e}",
                            "elapsed_h": f"{(now - t_start) / 3600:.2f}"})
                run_loss, run_n, t_log = 0.0, 0, now

            if done < steps_per_epoch:
                if state["global_step"] % EVAL_EVERY == 0:
                    run_validation(epoch, done, last_loss)
                if state["global_step"] % SAVE_EVERY == 0:
                    save_latest(epoch, done)
                    log(f"  >> saved latest.pt (epoch {epoch + 1}, step {done})")
                if out_of_time():
                    save_latest(epoch, done)
                    log(f"  >> time budget used up, saved latest.pt at epoch {epoch + 1} step {done}. "
                        f"Re-run in a new session to continue.")
                    return False

        # end of epoch
        log(f"===== epoch {epoch + 1}/{EPOCHS} finished =====")
        run_validation(epoch, steps_per_epoch, last_loss)
        save_latest(epoch + 1, 0)
        log(f"  >> epoch checkpoint saved (latest.pt now points to start of epoch {epoch + 2})")
        try:     # if this monitor fails, don't kill the training run
            score, samples = val_gen_score(model, tok, val_w, device, n=32, **MONITOR_DECODE)
            log(f"  >> greedy+rep1.1+ngram3 ROUGE-L on 32 val windows: {score:.4f}")
            log(f"  >> sample reference : {samples[0][0][:300]}")
            log(f"  >> sample generated : {samples[0][1][:300]}")
            append_log({"time": time.strftime("%H:%M:%S"), "kind": "gen", "epoch": epoch + 1,
                        "global_step": state["global_step"], "rouge_l": f"{score:.4f}",
                        "elapsed_h": f"{(time.time() - t_start) / 3600:.2f}"})
        except Exception as e:
            log(f"  >> generation monitor skipped: {type(e).__name__}: {e}")
        model.train()
        state["epoch"], state["step_in_epoch"] = epoch + 1, 0
        if epoch + 1 < EPOCHS and out_of_time():
            log("  >> time budget used up after this epoch; re-run in a new session to continue.")
            return False
    return True


# predict on the test set
def load_test_prompts():
    # test file -> {id: {"text": prompt}}, like the starter code
    token = hf_token()
    test_file_path = None
    for name in (TEST_FILE, TEST_FILE_ALT):
        try:
            test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=name,
                                             repo_type="dataset", token=token)
            break
        except Exception as e:
            log(f"could not download {name}: {type(e).__name__}")
    if test_file_path is None:               # last resort: any json in the Test/ folder
        found = sorted(f for f in list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=token)
                       if f.startswith("Phase_1_Pretraining/Test/") and f.endswith(".json"))
        if not found:
            raise FileNotFoundError("no test json found in the Hugging Face dataset")
        test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=found[0],
                                         repo_type="dataset", token=token)
    log(f"Test file: {test_file_path}")
    with open(test_file_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)
    log(f"Number of test examples: {len(test_data)}")
    return test_data


DECODE_CONFIGS = {
    "spec_t0.5_k40_p0.9_rep1.1_ng3": dict(temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
                                          rep_penalty=REPETITION_PENALTY, no_repeat_ngram=NO_REPEAT_NGRAM_SIZE),
    "greedy_rep1.1_ng3": dict(temperature=0.0, rep_penalty=1.1, no_repeat_ngram=3),
    "greedy_rep1.1": dict(temperature=0.0, rep_penalty=1.1),
    "greedy_rep1.2_ng3": dict(temperature=0.0, rep_penalty=1.2, no_repeat_ngram=3),
    "t0.7_p0.9_rep1.1_ng3": dict(temperature=0.7, top_p=0.9, rep_penalty=1.1, no_repeat_ngram=3),
}
MONITOR_DECODE = dict(temperature=0.0, rep_penalty=1.1, no_repeat_ngram=3)


def predict(model, tok, val_w, device):
    prompts = load_test_prompts()
    if not prompts:
        log("no test prompts available - skipping prediction")
        return
    torch.manual_seed(SEED)
    # choose the decoding recipe on val windows
    results = {}
    for name, kw in DECODE_CONFIGS.items():
        score, _ = val_gen_score(model, tok, val_w, device, n=96, bs=32, **kw)
        results[name] = score
        log(f"  decode {name:>32s}: val ROUGE-L {score:.4f}")
    best_name = max(results, key=results.get)
    log(f"using decoding config: {best_name}")
    kw = DECODE_CONFIGS[best_name]
    atomic_json({"scores": results, "chosen": best_name}, os.path.join(CKPT_DIR, "decoding_sweep.json"))

    keys = list(prompts.keys())
    enc = []
    for k in keys:
        ids = tok.encode(prompts[k]["text"])[-PROMPT_TOKENS:]       # keep the end of the prompt
        enc.append(ids if ids else [tok.eos_id])
    lens = [len(e) for e in enc]
    order = sorted(range(len(keys)), key=lambda i: lens[i])         # group similar lengths, less padding
    preds_by_key = {}
    bs = 32
    for i in range(0, len(order), bs):
        batch = order[i:i + bs]
        ids, mask = make_prompt_batch([enc[j] for j in batch], tok.pad_id, device)
        gen = generate_ids(model, ids, mask, **kw).cpu().tolist()
        for j, g in zip(batch, gen):
            preds_by_key[str(keys[j])] = {"text": to_words(g, tok)}
        log(f"  generated {min(i + bs, len(order))}/{len(order)}")
    preds = {str(k): preds_by_key[str(k)] for k in keys}             # keep the official id order
    log(f"prompt token lengths: mean {np.mean(lens):.0f}, max {max(lens)} (model keeps the last {PROMPT_TOKENS})")

    short = sum(1 for v in preds.values() if len(v["text"].split()) < TARGET_WORDS)
    if short:
        log(f"warning: {short} continuations have fewer than {TARGET_WORDS} words")
    assert set(preds) == {str(k) for k in prompts}, "prediction ids differ from test ids"

    # prediction.json and submission.zip, same format as the starter code
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)
    log(f"prediction.json saved: {PREDICTION_PATH}")
    with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(PREDICTION_PATH, arcname="prediction.json")      # has to be at the root of the zip
    log(f"Codabench submission created: {SUBMISSION_ZIP} ({len(preds)} predictions)")


# main
def find_best_weights():
    # best.pt (lowest val loss) of this model, in our dir or any attached kaggle input
    best = None
    for d in walk_find("best_meta.json", [CKPT_DIR, INPUT_ROOT]):
        meta = read_json(os.path.join(d, "best_meta.json"))
        if not meta or meta.get("tag") != MODEL_TAG or not os.path.exists(os.path.join(d, "best.pt")):
            continue
        if best is None or meta["val_loss"] < best[1]["val_loss"]:
            best = (d, meta)
    return best


def load_weights_for_prediction(model):
    best = find_best_weights()
    if best is not None:
        d, meta = best
        ckpt = torch.load(os.path.join(d, "best.pt"), map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model"])
        log(f"predicting with best.pt from {d}: val loss {meta['val_loss']:.4f} "
            f"(epoch {meta['epoch'] + 1}, global step {meta['global_step']})")
        return
    prev = find_previous()
    if prev is None:
        raise FileNotFoundError("no best.pt or latest.pt of this model found - attach the previous run's output")
    ckpt = torch.load(os.path.join(prev[0], "latest.pt"), map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])
    log(f"no best.pt found, predicting with latest.pt from {prev[0]}")


# decoding v2
# - beam search / greedy with a vectorised no repeat ngram (no python loop per row)
# - mbr decoding (pick the sampled candidate that agrees most with the other samples)
# - v3: same mbr idea but agreement is measured with bertscore itself ("mbr_bs"), the graded metric
# - makes sure we get 128 words (keeps generating for rows that came out short)
# - val sweep, so the best recipe is picked with numbers
# - main_decode() loads best.pt, sweeps, writes prediction.json + submission.zip
# reuses start_generation, next_logits, make_prompt_batch, split_window, to_words,
# clean_text, rouge_l_f1, load_test_prompts, load_weights_for_prediction
import subprocess
from collections import Counter

# settings
DECODE_MODE = "auto"        # auto = sweep on val windows and use the best, or greedy / beam / mbr / mbr_bs
QUICK_TEST = False          # True = quick smoke test (32 val windows, 16 test prompts, no zip)
BEAM_WIDTH = 4
NO_REPEAT_NGRAM_V2 = 4      # in bpe tokens (4-5 tokens is about 2-3 words)
MBR_SAMPLES = 12            # samples per prompt for mbr
MBR_TEMPERATURE = 0.8
MBR_TOP_P = 0.95
MBR_W_UNI = 0.7             # mbr score = 0.7 * unigram f1 + 0.3 * bigram f1
BS_MBR_SAMPLES = 24         # samples per prompt for the bertscore mbr
EMBED_MODEL = "roberta-large"   # model bert-score uses for lang="en", change it if the grader uses another one
EMBED_LAYER = 17            # layer bert-score uses in roberta-large
VAL_SWEEP_N = 128           # val windows used to compare recipes
ROW_BUDGET = 96             # how many sequences to decode at once, lower it (like 48) on cuda oom
EXTEND_TOKENS = 192         # extra tokens to generate when a continuation has under 128 words
EXTEND_ROUNDS = 3


# low level batched decoding with kv cache
def _expand(x, k):
    return x.repeat_interleave(k, dim=0) if k > 1 else x


def _sample_from(logits, temperature, top_k, top_p):
    logits = logits / temperature
    if top_k > 0:
        kk = min(top_k, logits.size(-1))
        kth = torch.topk(logits, kk).values[:, -1, None]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        probs = sl.softmax(-1)
        drop = (probs.cumsum(-1) - probs) > top_p
        sl = sl.masked_fill(drop, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def _block_ngrams(logits, seq, t, n, pad_id):
    # ban every token that would complete an n-gram already in seq[:, :t], all rows at once
    if n < 2 or t < n:
        return
    s = seq[:, :t]
    ctx = s[:, t - n + 1:]                                   # (R, n-1) the last n-1 tokens
    wins = s.unfold(1, n - 1, 1)[:, : t - n + 1]             # (R, t-n+1, n-1) earlier windows (not the context itself)
    match = (wins == ctx[:, None, :]).all(-1)                # (R, t-n+1)
    nxt = s[:, n - 1:]                                       # token that followed each window
    banned = torch.where(match, nxt, torch.full_like(nxt, pad_id))   # pad is already banned so it's a harmless filler
    logits.scatter_(1, banned, float("-inf"))


@torch.no_grad()
def decode_batch(model, prompt, prompt_mask, max_new=MAX_NEW_TOKENS, mode="beam", beam=4, n=1,
                 temperature=0.8, top_k=0, top_p=1.0, rep_penalty=1.0, no_repeat=0):
    # mode="beam": beam search with `beam` beams (beam=1 is greedy), returns (B, beam, max_new) with the best beam first
    # mode="sample": n independent samples per prompt, returns (B, n, max_new)
    # the prompt is prefilled once and its kv cache is copied to the beams / samples
    # eos, pad and unk are banned so every hypothesis has exactly max_new tokens (same length, so no length penalty needed)
    model.eval()
    cfg = model.cfg
    B, W = prompt.shape
    dev = prompt.device
    K = beam if mode == "beam" else n
    R = B * K
    seq = torch.zeros(R, max_new, dtype=torch.long, device=dev)
    with amp_ctx():
        cache, logits, lengths, valid = start_generation(model, prompt, prompt_mask, max_new)
        cache = [{"k": _expand(c["k"], K), "v": _expand(c["v"], K)} for c in cache]
        logits, lengths, valid = _expand(logits, K), _expand(lengths, K), _expand(valid, K)
        if mode == "beam":
            beam_scores = torch.zeros(B, K, device=dev)
            beam_scores[:, 1:] = -1e9                         # step 0: all beams are the same, only keep beam 0
            base = (torch.arange(B, device=dev) * K)[:, None]
            ident = torch.arange(R, device=dev)
        for t in range(max_new):
            logits = logits.float()
            logits[:, cfg.eos_id] = float("-inf")
            logits[:, cfg.pad_id] = float("-inf")
            logits[:, cfg.unk_id] = float("-inf")
            if rep_penalty != 1.0 and t > 0:
                prev = seq[:, :t]
                sc = logits.gather(1, prev)
                sc = torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty)
                logits.scatter_(1, prev, sc)
            _block_ngrams(logits, seq, t, no_repeat, cfg.pad_id)
            if mode == "beam":
                V = logits.size(-1)
                logp = torch.log_softmax(logits, dim=-1)
                cand = (beam_scores.view(R, 1) + logp).view(B, K * V)
                top, idx = cand.topk(K, dim=1)
                parent = torch.div(idx, V, rounding_mode="floor")
                tok = idx % V
                flat_parent = (base + parent).view(-1)
                beam_scores = top
                if not torch.equal(flat_parent, ident):       # beams got reordered, so reorder the history and kv cache too
                    seq = seq.index_select(0, flat_parent)
                    for c in cache:
                        c["k"] = c["k"].index_select(0, flat_parent)
                        c["v"] = c["v"].index_select(0, flat_parent)
                tok = tok.view(-1)
            else:
                tok = _sample_from(logits, temperature, top_k, top_p)
            seq[:, t] = tok
            if t < max_new - 1:
                logits = next_logits(model, tok, t, W, cache, lengths, valid)
    return seq.view(B, K, max_new)


# mbr: pick the candidate that agrees most with the others
def _grams(words):
    return Counter(words), Counter(zip(words, words[1:]))


def _f1(a, b):
    if not a or not b:
        return 0.0
    ov = sum((a & b).values())
    if ov == 0:
        return 0.0
    p, r = ov / sum(b.values()), ov / sum(a.values())
    return 2 * p * r / (p + r)


def _utility(fa, fb):
    return MBR_W_UNI * _f1(fa[0], fb[0]) + (1.0 - MBR_W_UNI) * _f1(fa[1], fb[1])


def n_words(ids, tok):
    return len(clean_text(tok.decode(ids)).split())


def _first_full(cands, tok):
    # first candidate with at least 128 words, else just the first one
    for c in cands:
        if n_words(c, tok) >= TARGET_WORDS:
            return c
    return cands[0]


def _mbr_choose(best_beam, samples, tok):
    # hypotheses = best beam + all samples, pseudo references = the samples
    # (the beams are near duplicates and would just vote for each other)
    # candidates with under 128 words are only used if nothing else exists
    cands = [best_beam] + list(samples)
    texts = [to_words(c, tok) for c in cands]
    words = [t.lower().split() for t in texts]
    full = [i for i, w in enumerate(words) if len(w) >= TARGET_WORDS] or list(range(len(cands)))
    feats = [_grams(w) for w in words]
    refs = list(range(1, len(cands)))
    best_i, best_s = full[0], -1.0
    for i in full:
        others = [j for j in refs if j != i]
        if not others:
            continue
        s = sum(_utility(feats[i], feats[j]) for j in others) / len(others)
        if s > best_s:
            best_i, best_s = i, s
    return cands[best_i]


# mbr with bertscore as the score
class BertScorer:
    # bertscore style greedy token matching (no idf, special tokens have no weight)
    # each text is encoded once, then any two texts in the pool are compared with a matrix product

    def __init__(self, name, layer, device):
        from transformers import AutoModel, AutoTokenizer
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        model = AutoModel.from_pretrained(name)
        model.encoder.layer = nn.ModuleList(model.encoder.layer[:layer])      # bert-score also stops at this layer
        if device.type == "cuda":
            model = model.half()
        self.model = model.to(device).eval()

    @torch.no_grad()
    def embed(self, texts, batch_size=32):
        # one (n_tokens, dim) tensor of unit length vectors per text
        out = []
        for s in range(0, len(texts), batch_size):
            enc = self.tokenizer(texts[s:s + batch_size], return_tensors="pt", padding=True, truncation=True,
                                 max_length=512, return_special_tokens_mask=True)
            special = enc.pop("special_tokens_mask").bool().to(self.device)
            enc = enc.to(self.device)
            hidden = F.normalize(self.model(**enc).last_hidden_state.float(), dim=-1)
            keep = enc["attention_mask"].bool() & ~special
            out.extend(hidden[r][keep[r]] for r in range(hidden.size(0)))
        return out

    @staticmethod
    def pairwise_f1(embs):
        # (n, n) matrix, [i, j] = bertscore f1 between text i and text j
        n = len(embs)
        width = max(1, max(e.size(0) for e in embs))
        grid = embs[0].new_zeros(n, width, embs[0].size(1))
        valid = torch.zeros(n, width, dtype=torch.bool, device=grid.device)
        for i, e in enumerate(embs):
            grid[i, :e.size(0)] = e
            valid[i, :e.size(0)] = True
        sim = torch.einsum("ild,jmd->ijlm", grid, grid)             # cosine similarity of every token pair
        sim.masked_fill_(~valid[None, :, None, :], -2.0)            # padding of text j never matches
        best = sim.max(dim=-1).values                               # (i, j, l): best match in j for token l of i
        sizes = valid.sum(1).clamp(min=1).float()
        prec = (best * valid[:, None, :]).sum(-1) / sizes[:, None]  # i as candidate, j as reference
        return 2 * prec * prec.T / (prec + prec.T).clamp(min=1e-8)  # recall of [i, j] is the precision of [j, i]


_scorer_state = {"scorer": None, "failed": False}


def get_scorer():
    # the shared BertScorer, loaded on first use, None if the embedding model can't be loaded
    st = _scorer_state
    if st["scorer"] is None and not st["failed"]:
        try:
            dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            st["scorer"] = BertScorer(EMBED_MODEL, EMBED_LAYER, dev)
            log(f"MBR scorer ready: {EMBED_MODEL}, layer {EMBED_LAYER}, on {dev}")
        except Exception as e:
            st["failed"] = True
            log(f"  (BERTScore-MBR unavailable: {type(e).__name__}: {str(e)[:80]})")
    return st["scorer"]


def _mbr_choose_bs(best_beam, samples, tok, scorer):
    # like _mbr_choose but agreement between candidates is bertscore f1. the beam is only a candidate (never votes),
    # the samples are candidates and voters. short candidates are only used if nothing else exists
    cands = [best_beam] + list(samples)
    texts = [to_words(c, tok) for c in cands]
    full = [i for i, t in enumerate(texts) if len(t.split()) >= TARGET_WORDS] or list(range(len(cands)))
    f1 = scorer.pairwise_f1(scorer.embed(texts))
    f1.fill_diagonal_(0.0)
    f1[:, 0] = 0.0
    votes = torch.full((len(cands),), float(len(samples)), device=f1.device)
    votes[1:] -= 1                                                  # a sample doesn't vote for itself
    score = (f1.sum(1) / votes.clamp(min=1)).tolist()
    return cands[max(full, key=lambda i: score[i])]


# run one recipe over many prompts
def _decode_group(model, tok, ids, mask, c):
    kind = c["kind"]
    common = dict(max_new=MAX_NEW_TOKENS, rep_penalty=c.get("rep", 1.0), no_repeat=c.get("ng", 0))
    if kind == "sample":
        seqs = decode_batch(model, ids, mask, mode="sample", n=c["n"], temperature=c["temperature"],
                            top_k=c.get("top_k", 0), top_p=c.get("top_p", 1.0), **common).cpu().tolist()
        return [_first_full(s, tok) for s in seqs]
    beams = decode_batch(model, ids, mask, mode="beam", beam=c["beam"], **common).cpu().tolist()
    if kind == "beam":
        return [_first_full(b, tok) for b in beams]
    samples = decode_batch(model, ids, mask, mode="sample", n=c["n"], temperature=c["temperature"],
                           top_k=c.get("top_k", 0), top_p=c.get("top_p", 1.0), **common).cpu().tolist()
    if kind == "mbr_bs":
        scorer = get_scorer()
        return [_mbr_choose_bs(beams[b][0], samples[b], tok, scorer) for b in range(len(beams))]
    return [_mbr_choose(beams[b][0], samples[b], tok) for b in range(len(beams))]


def _safe_batch(fn, items):
    # run fn(items), on cuda oom split the batch in half and retry
    try:
        return fn(items)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(items) == 1:
            raise
        h = len(items) // 2
        log(f"  CUDA OOM, splitting batch of {len(items)}")
        return _safe_batch(fn, items[:h]) + _safe_batch(fn, items[h:])


def _extend_until_target(model, tok, prompt_ids, gen_ids, device):
    # keep generating (beam search) until the continuation has at least 128 words
    for _ in range(EXTEND_ROUNDS):
        if n_words(gen_ids, tok) >= TARGET_WORDS:
            break
        ctx = (prompt_ids + gen_ids)[-PROMPT_TOKENS:]
        ids, mask = make_prompt_batch([ctx], tok.pad_id, device)
        more = decode_batch(model, ids, mask, max_new=EXTEND_TOKENS, mode="beam", beam=BEAM_WIDTH,
                            no_repeat=NO_REPEAT_NGRAM_V2)[0, 0].tolist()
        gen_ids = gen_ids + more
    return gen_ids


def run_recipe(model, tok, enc, c, device, name=""):
    # enc is a list of prompt token id lists, returns the chosen continuation token ids for each prompt (same order)
    order = sorted(range(len(enc)), key=lambda i: len(enc[i]))      # group similar lengths, less padding
    rows_pp = c["beam"] if c["kind"] == "beam" else (c["n"] if c["kind"] == "sample" else max(c["beam"], c["n"]))
    bs = max(1, min(32, ROW_BUDGET // rows_pp))
    out = [None] * len(enc)

    def work(idxs):
        ids, mask = make_prompt_batch([enc[i] for i in idxs], tok.pad_id, device)
        return list(zip(idxs, _decode_group(model, tok, ids, mask, c)))

    t0 = time.time()
    for s in range(0, len(order), bs):
        for i, g in _safe_batch(work, order[s:s + bs]):
            out[i] = g
        done = min(s + bs, len(order))
        if (s // bs) % 10 == 0 or done == len(order):
            el = time.time() - t0
            log(f"  [{name}] {done}/{len(order)} prompts | {fmt_time(el)} elapsed | eta {fmt_time(el / done * (len(order) - done))}")
    n_short = 0
    for i in range(len(out)):
        if n_words(out[i], tok) < TARGET_WORDS:
            n_short += 1
            out[i] = _extend_until_target(model, tok, enc[i], out[i], device)
    if n_short:
        log(f"  [{name}] {n_short} continuations were < {TARGET_WORDS} words and were extended")
    return out


# configs and val sweep
def recipe_table():
    ng = NO_REPEAT_NGRAM_V2
    table = {
        "baseline_sample": dict(kind="sample", n=1, temperature=0.5, top_k=40, top_p=0.9, rep=1.1, ng=3),  # old recipe
        "greedy": dict(kind="beam", beam=1, ng=ng),
        "beam": dict(kind="beam", beam=BEAM_WIDTH, ng=ng),
        "beam8": dict(kind="beam", beam=8, ng=ng),
        "mbr": dict(kind="mbr", beam=BEAM_WIDTH, n=MBR_SAMPLES, temperature=MBR_TEMPERATURE,
                    top_k=0, top_p=MBR_TOP_P, rep=1.0, ng=ng),
    }
    if get_scorer() is not None:
        for name, temp in (("mbr_bs", 0.7), ("mbr_bs_hot", 1.0)):
            table[name] = dict(kind="mbr_bs", beam=BEAM_WIDTH, n=BS_MBR_SAMPLES, temperature=temp,
                               top_k=0, top_p=MBR_TOP_P, rep=1.0, ng=ng)
    return table


def bertscore_f1(hyps, refs, device):
    # mean bertscore f1 if the bert-score package (and its model) loads, else None
    try:
        try:
            import bert_score  # noqa: F401
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=False)
        from bert_score import score as bs_score
        _, _, f = bs_score(hyps, refs, lang="en", device=str(device), batch_size=16, verbose=False)
        return float(f.mean())
    except Exception as e:
        log(f"  (BERTScore unavailable: {type(e).__name__}: {str(e)[:80]})")
        return None


def val_sweep(model, tok, val_w, device, n):
    idx = np.linspace(0, len(val_w) - 1, min(n, len(val_w))).astype(int)      # spread over all four sources
    rows = val_w[idx].astype(np.int64).tolist()
    splits = [split_window(r) for r in rows]
    enc = [r[:s] for r, s in zip(rows, splits)]
    refs = [to_words(r[s:s + MAX_NEW_TOKENS], tok) for r, s in zip(rows, splits)]
    results = {}
    for name, c in recipe_table().items():
        t0 = time.time()
        gens = run_recipe(model, tok, enc, c, device, name)
        hyps = [to_words(g, tok) for g in gens]
        rl = float(np.mean([rouge_l_f1(r, h) for r, h in zip(refs, hyps)]))
        ng = float(np.mean([_utility(_grams(r.lower().split()), _grams(h.lower().split())) for r, h in zip(refs, hyps)]))
        bs = bertscore_f1(hyps, refs, device)
        wc = float(np.mean([len(h.split()) for h in hyps]))
        results[name] = {"bertscore": bs, "ngram_f1": ng, "rougeL": rl, "avg_words": wc, "sec": time.time() - t0}
        log(f"SWEEP {name:>16s} | BERTScore {('%.4f' % bs) if bs is not None else '  n/a '} | ngram-F1 {ng:.4f} | "
            f"ROUGE-L {rl:.4f} | words {wc:.1f} | {time.time() - t0:.0f}s")
    key = (lambda k: results[k]["bertscore"]) if all(v["bertscore"] is not None for v in results.values()) \
        else (lambda k: results[k]["ngram_f1"])
    best = max(results, key=key)
    return best, results


def load_cached_val_windows():
    # only load a val cache that belongs to this clean corpus run, never an arbitrary val_windows.npy
    expected_kind = "clean_bpe_training_v1"

    for d in walk_find(
        "windows_manifest.json",
        [CACHE_DIR, INPUT_ROOT],
    ):
        try:
            with open(
                os.path.join(d, "windows_manifest.json")
            ) as f:
                man = json.load(f)

            sig = man.get("sig", {})

            if sig.get("kind") != expected_kind:
                continue

            va = np.load(
                os.path.join(d, "val_windows.npy")
            )

            if (
                va.ndim == 2
                and va.shape[1] == CONTEXT_LENGTH + 1
            ):
                log(
                    f"clean-corpus validation windows loaded "
                    f"from {d}: {va.shape}"
                )
                return va

        except Exception as e:
            log(
                f"could not read validation cache in {d}: {e}"
            )

    return None


@torch.no_grad()
def check_decoder(model, tok, val_w, device):
    # beam=1 of the new decoder should match the old greedy generate_ids (fp16 near ties can differ a bit)
    rows = val_w[:2].astype(np.int64).tolist()
    ids, mask = make_prompt_batch([r[:split_window(r)] for r in rows], tok.pad_id, device)
    old = generate_ids(model, ids, mask, max_new=48, temperature=0.0)
    new = decode_batch(model, ids, mask, max_new=48, mode="beam", beam=1)[:, 0]
    agree = (old == new).float().mean().item()
    log(f"decoder check: greedy agreement with the old path = {agree:.3f}" + ("" if agree > 0.9 else "  <-- WARNING"))


# predict on the test set
def predict_v2(model, tok, val_w, device):
    prompts = load_test_prompts()
    keys = list(prompts.keys())
    if QUICK_TEST:
        keys = keys[:16]
    # pick the recipe
    if DECODE_MODE == "auto" and val_w is not None:
        best, results = val_sweep(model, tok, val_w, device, 32 if QUICK_TEST else VAL_SWEEP_N)
        atomic_json({"chosen": best, "results": results}, os.path.join(CKPT_DIR, "decoding_sweep_v2.json"))
        log(f"chosen recipe: {best}")
        recipe = recipe_table()[best]
    else:
        name = "beam" if DECODE_MODE == "auto" else DECODE_MODE
        if DECODE_MODE == "auto":
            log("no validation windows found -> falling back to plain beam search")
        recipe = recipe_table()[name]
        log(f"recipe: {name}")
    # decode the test prompts
    enc = []
    for k in keys:
        ids = tok.encode(prompts[k]["text"])[-PROMPT_TOKENS:]
        enc.append(ids if ids else [tok.eos_id])
    gens = run_recipe(model, tok, enc, recipe, device, "test")
    preds = {str(k): {"text": to_words(g, tok)} for k, g in zip(keys, gens)}
    short = sum(1 for v in preds.values() if len(v["text"].split()) < TARGET_WORDS)
    lens = [len(v["text"].split()) for v in preds.values()]
    log(f"prediction word counts: min/avg/max = {min(lens)}/{np.mean(lens):.1f}/{max(lens)} | below {TARGET_WORDS}: {short}")
    # write the files
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if QUICK_TEST:
        path = os.path.join(OUTPUT_DIR, "prediction_QUICKTEST.json")
        with open(path, "w", encoding="utf-8") as f:
            json.dump(preds, f, ensure_ascii=False, indent=2)
        log(f"quick test only - wrote {path}, no submission zip")
        return
    assert set(preds) == {str(k) for k in prompts}, "prediction ids differ from test ids"
    preds = {str(k): preds[str(k)] for k in prompts}                       # keep the official id order
    with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(PREDICTION_PATH, arcname="prediction.json")                # has to be at the root of the zip
    log(f"saved {PREDICTION_PATH} and {SUBMISSION_ZIP} ({len(preds)} predictions)")


def main_decode():
    # predict only, no training or corpus streaming. needs best.pt (ideally also the window cache) attached as input
    hf_login()
    os.makedirs(CKPT_DIR, exist_ok=True)
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    if use_cuda:
        AMP["enabled"] = True
        AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        log(f"GPU: {torch.cuda.get_device_name()} | autocast dtype: {AMP['dtype']}")
    else:
        log("WARNING: no GPU visible, this will be extremely slow")
    tok = load_tokenizer()
    assert tok.vocab_size == VOCAB_SIZE, f"tokenizer has {tok.vocab_size} tokens, expected {VOCAB_SIZE}"
    cfg = ModelConfig(vocab_size=tok.vocab_size, pad_id=tok.pad_id, unk_id=tok.unk_id, eos_id=tok.eos_id)
    model = DecoderLM(cfg).to(device)
    load_weights_for_prediction(model)
    model.eval()
    val_w = load_cached_val_windows()
    if val_w is not None:
        check_decoder(model, tok, val_w, device)
    predict_v2(model, tok, val_w, device)
    log("done.")



def main():
    t_start = time.time()
    hf_login()
    os.makedirs(CKPT_DIR, exist_ok=True)
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    if use_cuda:
        major = torch.cuda.get_device_capability()[0]
        AMP["enabled"] = True
        AMP["dtype"] = torch.bfloat16 if major >= 8 else torch.float16
        log(f"GPU: {torch.cuda.get_device_name()} | autocast dtype: {AMP['dtype']}")
    else:
        log("WARNING: no GPU visible, this will be extremely slow")

    tok = load_tokenizer()
    assert tok.vocab_size == VOCAB_SIZE, f"tokenizer has {tok.vocab_size} tokens, expected {VOCAB_SIZE}"
    train_w, val_w = get_windows(tok)
    log(f"train windows {len(train_w):,} ({len(train_w) * CONTEXT_LENGTH / 1e6:.0f}M tokens/epoch) | val windows {len(val_w):,}")

    cfg = ModelConfig(vocab_size=tok.vocab_size, pad_id=tok.pad_id, unk_id=tok.unk_id, eos_id=tok.eos_id)
    model = DecoderLM(cfg).to(device)
    n_params = count_params(model)
    log(f"model parameters: {n_params / 1e6:.1f}M")
    assert 50e6 <= n_params <= 300e6, "parameter count outside the 50M-300M limit"
    if RUN_SELF_TEST:
        self_test(model, val_w, device)

    finished = True if PREDICT_ONLY else train(model, tok, train_w, val_w, device, t_start)

    if finished and RUN_PREDICTION:
        load_weights_for_prediction(model)
        predict(model, tok, val_w, device)
    log("done.")


if __name__ == "__main__":
    # normal run: train or resume, then make the submission
    # set PREDICT_ONLY=True above to only predict
    main()